## Vallius, Shi, Novikov et al. 2026.
### FIG 2C and Ext Fig 3D,E plotting

In [1]:
import sys
import os
import anndata as ad
import pandas as pd
import scanpy as sc
import re
import numpy as np
import seaborn as sns; sns.set(color_codes=True)
import scimap as sm  
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import pearsonr
import napari

import itertools
from scipy.stats import mannwhitneyu, levene

Running SCIMAP  2.1.0


### Single-marker, progression stage plots (Fig 2C, Ext Fig 3E)

#### MART1

In [79]:
# final script used for Fig2C KI67
def plot_proportions_by_celltype(
    df,
    region_key='ROI_major_category',
    celltype_key='lineage',
    sample_key='MEL_id',
    region_order=None,
    box_palette=None,
    savedir='celltype_proportion_plots',
    figsize=(3, 3)   # fixed figure size
):
    """
    Generates boxplots of proportions for each cell type, with:
      - Mann–Whitney U and Levene tests between region pairs (only p<=0.05 shown)
      - Samples with no ROI/celltype combinations filled with 0 for proportion/count/region_total.
      - Fixed figure size and fixed subplot margins so x-axis width is constant.
    """
    os.makedirs(savedir, exist_ok=True)
    sns.set_style("whitegrid")

    # ---------- EXPAND DF: per-sample, per-ROI × all cell types; fill missing with 0 ----------
    all_celltypes = df[celltype_key].unique()
    expanded_parts = []

    for sample_id, sample_df in df.groupby(sample_key):
        sample_rois = sample_df[region_key].dropna().unique()

        full_index = pd.MultiIndex.from_product(
            [sample_rois, all_celltypes],
            names=[region_key, celltype_key]
        )

        sample_expanded = (
            sample_df
            .set_index([region_key, celltype_key])
            .reindex(full_index)
            .reset_index()
        )

        sample_expanded[sample_key] = sample_id

        if 'proportion' in sample_expanded.columns:
            sample_expanded['proportion'] = sample_expanded['proportion'].fillna(0)
        if 'count' in sample_expanded.columns:
            sample_expanded['count'] = sample_expanded['count'].fillna(0).astype(int)
        if 'region_total' in sample_expanded.columns:
            sample_expanded['region_total'] = sample_expanded['region_total'].fillna(0).astype(int)

        expanded_parts.append(sample_expanded)

    df_expanded = pd.concat(expanded_parts, ignore_index=True)

    # ---------- PLOTTING ----------
    unique_celltypes = df_expanded[celltype_key].unique()
    print(f"\nGenerating {len(unique_celltypes)} boxplots (one for each cell type).")

    for cell_type in unique_celltypes:
        plot_df = df_expanded[df_expanded[celltype_key] == cell_type].copy()
        if plot_df.empty:
            print(f"Skipping plot for {cell_type}: No data available.")
            continue

        # fixed-size figure for every cell type
        fig, ax = plt.subplots(figsize=figsize)

        # fixed margins so the axes width is constant across plots
        # (adjust these numbers if you want more/less space for labels)
        fig.subplots_adjust(left=0.22, right=0.95, bottom=0.35, top=0.88)

        # ----- region order -----
        if region_order is None:
            this_region_order = sorted(plot_df[region_key].dropna().unique())
        else:
            this_region_order = region_order

        palette_to_use = box_palette if box_palette is not None else 'Set2'

        # ----- boxplot -----
        ax = sns.boxplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            palette=palette_to_use,
            width=0.5,
            showfliers=False,
            linewidth=0.5,
            ax=ax
        )

        ylims = ax.get_ylim()

        # ----- stripplot -----
        ax = sns.stripplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            size=1,
            jitter=True,
            dodge=False,
            edgecolor='black',
            linewidth=0.5,
            ax=ax
        )

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        # ----- add n above each box -----
        stats = (
            plot_df
            .groupby(region_key)['proportion']
            .agg(count='size', ymax='max')
            .reindex(this_region_order)
        )

        tick_locs = ax.get_xticks()
        tick_labels = [t.get_text() for t in ax.get_xticklabels()]
        x_pos_map = dict(zip(tick_labels, tick_locs))

        for region, row in stats.iterrows():
            if pd.isna(row['count']):
                continue
            n = int(row['count'])
            if region not in x_pos_map:
                continue
            x = x_pos_map[region]
            y_raw = (row['ymax'] * 1.05) if pd.notnull(row['ymax']) else 0.0
            y = min(y_raw, 0.98)

            ax.text(
                x,
                y,
                f"n={n}",
                ha="center",
                va="bottom",
                fontsize=7,
                color="black"
            )

        # ----- Mann–Whitney U and Levene tests between each pair of regions -----
        lines = []
        region_pairs = list(itertools.combinations(this_region_order, 2))

        for r1, r2 in region_pairs:
            g1 = plot_df.loc[plot_df[region_key] == r1, 'proportion'].dropna()
            g2 = plot_df.loc[plot_df[region_key] == r2, 'proportion'].dropna()

            if len(g1) < 2 or len(g2) < 2:
                continue

            stat_mw, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
            stat_lev, p_lev = levene(g1, g2, center='median')

            pair_header_added = False

            if p_mw <= 0.05:
                lines.append(f"{r1} vs {r2}:")
                lines.append(f"  MWU p={p_mw:.2e}")
                pair_header_added = True

            if p_lev <= 0.05:
                if not pair_header_added:
                    lines.append(f"{r1} vs {r2}:")
                lines.append(f"  Lev p={p_lev:.2e}")

        if lines:
            ax.text(
                0.72, 0.98,
                "\n".join(lines),
                transform=ax.transAxes,
                va='top',
                ha='left',
                fontsize=6,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8)
            )

        # ----- aesthetics -----
        ax.set_title(f"{cell_type}", fontsize=10)
        ax.set_xlabel(region_key.capitalize(), fontsize=9)
        ax.set_ylabel(f"Proportion of {cell_type}", fontsize=9)
        ax.set(ylim=ylims)
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.yticks(fontsize=8)

        safe_cell_type = cell_type.replace(" ", "_").replace("-", "_")
        plot_filename = os.path.join(
            savedir,
            f"Fig2_suppl_{safe_cell_type}_proportions_boxplot-Breslow.pdf"
        )

        try:
            fig.savefig(plot_filename)  # no bbox_inches='tight'
            print(f"  Plot for {cell_type} saved to: {plot_filename}")
        except Exception as e:
            print(f"  Error saving plot for {cell_type}: {e}")

        plt.close(fig)

In [80]:
proportion_results=pd.read_csv(r"C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\supplementary materials\CyCIF_data_for_plotting\Fig2C-MART1-single-marker-merged-data_for-plotting.csv")

order=['normal','precursor','MIS','invasive']

custom_colors = {
    "normal":   "#4782b6",
    "precursor":"#278b44",
    "MIS":      "#ea9ec5",
    "invasive": "#f1634b"
}

plot_proportions_by_celltype(
    df=proportion_results,
    region_key='ROI_major_category',
    celltype_key='MART1pos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2215365903.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Error saving plot for MART1+: [Errno 13] Permission denied: 'celltype_proportion_plots\\Fig2_suppl_MART1+_proportions_boxplot-Breslow.pdf'


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2215365903.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for MART1- saved to: celltype_proportion_plots\Fig2_suppl_MART1__proportions_boxplot-Breslow.pdf


#### KI67

In [53]:
# final script used for Fig2C KI67
def plot_proportions_by_celltype(
    df,
    region_key='ROI_major_category',
    celltype_key='lineage',
    sample_key='MEL_id',
    region_order=None,
    box_palette=None,
    savedir='celltype_proportion_plots',
    figsize=(3, 3)   # fixed figure size
):
    """
    Generates boxplots of proportions for each cell type, with:
      - Mann–Whitney U and Levene tests between region pairs (only p<=0.05 shown)
      - Samples with no ROI/celltype combinations filled with 0 for proportion/count/region_total.
      - Fixed figure size and fixed subplot margins so x-axis width is constant.
    """
    os.makedirs(savedir, exist_ok=True)
    sns.set_style("whitegrid")

    # ---------- EXPAND DF: per-sample, per-ROI × all cell types; fill missing with 0 ----------
    all_celltypes = df[celltype_key].unique()
    expanded_parts = []

    for sample_id, sample_df in df.groupby(sample_key):
        sample_rois = sample_df[region_key].dropna().unique()

        full_index = pd.MultiIndex.from_product(
            [sample_rois, all_celltypes],
            names=[region_key, celltype_key]
        )

        sample_expanded = (
            sample_df
            .set_index([region_key, celltype_key])
            .reindex(full_index)
            .reset_index()
        )

        sample_expanded[sample_key] = sample_id

        if 'proportion' in sample_expanded.columns:
            sample_expanded['proportion'] = sample_expanded['proportion'].fillna(0)
        if 'count' in sample_expanded.columns:
            sample_expanded['count'] = sample_expanded['count'].fillna(0).astype(int)
        if 'region_total' in sample_expanded.columns:
            sample_expanded['region_total'] = sample_expanded['region_total'].fillna(0).astype(int)

        expanded_parts.append(sample_expanded)

    df_expanded = pd.concat(expanded_parts, ignore_index=True)

    # ---------- PLOTTING ----------
    unique_celltypes = df_expanded[celltype_key].unique()
    print(f"\nGenerating {len(unique_celltypes)} boxplots (one for each cell type).")

    for cell_type in unique_celltypes:
        plot_df = df_expanded[df_expanded[celltype_key] == cell_type].copy()
        if plot_df.empty:
            print(f"Skipping plot for {cell_type}: No data available.")
            continue

        # fixed-size figure for every cell type
        fig, ax = plt.subplots(figsize=figsize)

        # fixed margins so the axes width is constant across plots
        # (adjust these numbers if you want more/less space for labels)
        fig.subplots_adjust(left=0.22, right=0.95, bottom=0.35, top=0.88)

        # ----- region order -----
        if region_order is None:
            this_region_order = sorted(plot_df[region_key].dropna().unique())
        else:
            this_region_order = region_order

        palette_to_use = box_palette if box_palette is not None else 'Set2'

        # ----- boxplot -----
        ax = sns.boxplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            palette=palette_to_use,
            width=0.5,
            showfliers=False,
            linewidth=0.5,
            ax=ax
        )

        ylims = ax.get_ylim()

        # ----- stripplot -----
        ax = sns.stripplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            size=1,
            jitter=True,
            dodge=False,
            edgecolor='black',
            linewidth=0.5,
            ax=ax
        )

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        # ----- add n above each box -----
        stats = (
            plot_df
            .groupby(region_key)['proportion']
            .agg(count='size', ymax='max')
            .reindex(this_region_order)
        )

        tick_locs = ax.get_xticks()
        tick_labels = [t.get_text() for t in ax.get_xticklabels()]
        x_pos_map = dict(zip(tick_labels, tick_locs))

        for region, row in stats.iterrows():
            if pd.isna(row['count']):
                continue
            n = int(row['count'])
            if region not in x_pos_map:
                continue
            x = x_pos_map[region]
            y_raw = (row['ymax'] * 1.05) if pd.notnull(row['ymax']) else 0.0
            y = min(y_raw, 0.98)

            ax.text(
                x,
                y,
                f"n={n}",
                ha="center",
                va="bottom",
                fontsize=7,
                color="black"
            )

        # ----- Mann–Whitney U and Levene tests between each pair of regions -----
        lines = []
        region_pairs = list(itertools.combinations(this_region_order, 2))

        for r1, r2 in region_pairs:
            g1 = plot_df.loc[plot_df[region_key] == r1, 'proportion'].dropna()
            g2 = plot_df.loc[plot_df[region_key] == r2, 'proportion'].dropna()

            if len(g1) < 2 or len(g2) < 2:
                continue

            stat_mw, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
            stat_lev, p_lev = levene(g1, g2, center='median')

            pair_header_added = False

            if p_mw <= 0.05:
                lines.append(f"{r1} vs {r2}:")
                lines.append(f"  MWU p={p_mw:.2e}")
                pair_header_added = True

            if p_lev <= 0.05:
                if not pair_header_added:
                    lines.append(f"{r1} vs {r2}:")
                lines.append(f"  Lev p={p_lev:.2e}")

        if lines:
            ax.text(
                0.72, 0.98,
                "\n".join(lines),
                transform=ax.transAxes,
                va='top',
                ha='left',
                fontsize=6,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8)
            )

        # ----- aesthetics -----
        ax.set_title(f"{cell_type}", fontsize=10)
        ax.set_xlabel(region_key.capitalize(), fontsize=9)
        ax.set_ylabel(f"Proportion of {cell_type}", fontsize=9)
        ax.set(ylim=ylims)
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.yticks(fontsize=8)

        safe_cell_type = cell_type.replace(" ", "_").replace("-", "_")
        plot_filename = os.path.join(
            savedir,
            f"Fig2_suppl_{safe_cell_type}_proportions_boxplot-Breslow.pdf"
        )

        try:
            fig.savefig(plot_filename)  # no bbox_inches='tight'
            print(f"  Plot for {cell_type} saved to: {plot_filename}")
        except Exception as e:
            print(f"  Error saving plot for {cell_type}: {e}")

        plt.close(fig)

In [60]:
proportion_results=pd.read_csv(r"C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\supplementary materials\CyCIF_data_for_plotting\Fig2C-KI67-single-marker-merged-data_for-plotting.csv")

order=['normal','precursor','MIS','invasive']

custom_colors = {
    "normal":   "#4782b6",
    "precursor":"#278b44",
    "MIS":      "#ea9ec5",
    "invasive": "#f1634b"
}

plot_proportions_by_celltype(
    df=proportion_results,
    region_key='ROI_major_category',
    celltype_key='Ki67pos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\1260070920.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for Ki67+ saved to: celltype_proportion_plots\Fig2_suppl_Ki67+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\1260070920.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for Ki67- saved to: celltype_proportion_plots\Fig2_suppl_Ki67__proportions_boxplot-Breslow.pdf


#### NGFR

In [65]:
## Modified script to allow breaking down of the y-axis

def plot_proportions_by_celltype(
    df,
    region_key='ROI_major_category',
    celltype_key='lineage',
    sample_key='MEL_id',
    region_order=None,
    box_palette=None,
    savedir='celltype_proportion_plots',
    # --- broken y-axis control ---
    bottom_ylim=None,        # e.g. (0, 0.2)
    top_ylim=None,           # e.g. (0.6, 1.0)
    figsize_single=(3, 3),
    figsize_broken=(3, 3.8),
    broken_height_ratios=(3, 1)  # NEW: relative heights (top, bottom)
):
    """
    Generates boxplots of proportions for each cell type, with:
      - Mann–Whitney U and Levene tests between region pairs (only p<=0.05 shown)
      - Samples with no ROI/celltype combinations filled with 0 for proportion/count/region_total.
      - Optional broken y-axis when bottom_ylim and top_ylim are given.
      - Optionally control relative vertical size of top vs. bottom of broken axis.
    """
    os.makedirs(savedir, exist_ok=True)
    sns.set_style("whitegrid")

    # ---------- EXPAND DF ----------
    all_celltypes = df[celltype_key].unique()
    expanded_parts = []

    for sample_id, sample_df in df.groupby(sample_key):
        sample_rois = sample_df[region_key].dropna().unique()
        full_index = pd.MultiIndex.from_product(
            [sample_rois, all_celltypes],
            names=[region_key, celltype_key]
        )
        sample_expanded = (
            sample_df
            .set_index([region_key, celltype_key])
            .reindex(full_index)
            .reset_index()
        )
        sample_expanded[sample_key] = sample_id

        if 'proportion' in sample_expanded.columns:
            sample_expanded['proportion'] = sample_expanded['proportion'].fillna(0)
        if 'count' in sample_expanded.columns:
            sample_expanded['count'] = sample_expanded['count'].fillna(0).astype(int)
        if 'region_total' in sample_expanded.columns:
            sample_expanded['region_total'] = sample_expanded['region_total'].fillna(0).astype(int)

        expanded_parts.append(sample_expanded)

    df_expanded = pd.concat(expanded_parts, ignore_index=True)

    # ---------- PLOTTING ----------
    unique_celltypes = df_expanded[celltype_key].unique()
    print(f"\nGenerating {len(unique_celltypes)} boxplots (one for each cell type).")

    use_broken_axis = (bottom_ylim is not None) and (top_ylim is not None)

    for cell_type in unique_celltypes:
        plot_df = df_expanded[df_expanded[celltype_key] == cell_type].copy()
        if plot_df.empty:
            print(f"Skipping plot for {cell_type}: No data available.")
            continue

        # ----- region order -----
        if region_order is None:
            this_region_order = sorted(plot_df[region_key].dropna().unique())
        else:
            this_region_order = region_order

        palette_to_use = box_palette if box_palette is not None else 'Set2'

        # ---------- FIGURE / AXES SETUP ----------
        if use_broken_axis:
            fig, (ax_top, ax_bottom) = plt.subplots(
                2, 1,
                sharex=True,
                figsize=figsize_broken,
                gridspec_kw={
                    "height_ratios": broken_height_ratios,  # <--- use parameter
                    "hspace": 0.05
                }
            )
            axes_for_plotting = [ax_top, ax_bottom]
        else:
            fig, ax = plt.subplots(figsize=figsize_single)
            axes_for_plotting = [ax]

        # ---------- BOXPLOT + STRIPPLOT ----------
        for a in axes_for_plotting:
            sns.boxplot(
                data=plot_df,
                x=region_key,
                y='proportion',
                order=this_region_order,
                palette=palette_to_use,
                width=0.5,
                showfliers=False,
                linewidth=0.5,
                ax=a
            )
            sns.stripplot(
                data=plot_df,
                x=region_key,
                y='proportion',
                order=this_region_order,
                size=1,
                jitter=True,
                dodge=False,
                edgecolor='black',
                linewidth=0.5,
                ax=a
            )

        if not use_broken_axis:
            ylims = ax.get_ylim()

        # remove legend
        for a in axes_for_plotting:
            leg = a.get_legend()
            if leg is not None:
                leg.remove()

        # ---------- n ABOVE EACH BOX ----------
        stats = (
            plot_df
            .groupby(region_key)['proportion']
            .agg(count='size', ymax='max')
            .reindex(this_region_order)
        )

        ax_for_n = axes_for_plotting[0]

        tick_locs = ax_for_n.get_xticks()
        tick_labels = [t.get_text() for t in ax_for_n.get_xticklabels()]
        x_pos_map = dict(zip(tick_labels, tick_locs))

        if use_broken_axis:
            y_max_for_n = top_ylim[1]
        else:
            y_max_for_n = ylims[1]

        for region, row in stats.iterrows():
            if pd.isna(row['count']):
                continue
            n = int(row['count'])
            if region not in x_pos_map:
                continue
            x = x_pos_map[region]
            y_raw = (row['ymax'] * 1.05) if pd.notnull(row['ymax']) else 0.0
            y = min(y_raw, y_max_for_n * 0.98)

            ax_for_n.text(
                x,
                y,
                f"n={n}",
                ha="center",
                va="bottom",
                fontsize=7,
                color="black"
            )

        # ---------- STATS TEXT ----------
        lines = []
        region_pairs = list(itertools.combinations(this_region_order, 2))

        for r1, r2 in region_pairs:
            g1 = plot_df.loc[plot_df[region_key] == r1, 'proportion'].dropna()
            g2 = plot_df.loc[plot_df[region_key] == r2, 'proportion'].dropna()

            if len(g1) < 2 or len(g2) < 2:
                continue

            stat_mw, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
            stat_lev, p_lev = levene(g1, g2, center='median')

            pair_header_added = False

            if p_mw <= 0.05:
                lines.append(f"{r1} vs {r2}:")
                lines.append(f"  MWU p={p_mw:.2e}")
                pair_header_added = True

            if p_lev <= 0.05:
                if not pair_header_added:
                    lines.append(f"{r1} vs {r2}:")
                lines.append(f"  Lev p={p_lev:.2e}")

        if lines:
            ax_for_stats = axes_for_plotting[0]
            ax_for_stats.text(
                0.72, 0.98,
                "\n".join(lines),
                transform=ax_for_stats.transAxes,
                va='top',
                ha='left',
                fontsize=6,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8)
            )

        # ---------- Y-LIMITS / BROKEN AXIS AESTHETICS ----------
        if use_broken_axis:
            ax_top.set_ylim(top_ylim)
            ax_bottom.set_ylim(bottom_ylim)

            ax_top.spines['bottom'].set_visible(False)
            ax_bottom.spines['top'].set_visible(False)

            ax_top.tick_params(labeltop=False)
            ax_bottom.tick_params(labeltop=False)

            # diagonal break marks
            d = .015
            kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False)
            ax_top.plot((-d, +d), (-d, +d), **kwargs)
            ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)

            kwargs = dict(transform=ax_bottom.transAxes, color='k', clip_on=False)
            ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs)
            ax_bottom.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

            ax = ax_top  # use top axis for title/y-label
        else:
            ax.set(ylim=ylims)

        # ---------- LABELS / TITLES ----------
        ax.set_title(f"{cell_type}", fontsize=10)
        ax.set_ylabel(f"Proportion of {cell_type}", fontsize=9)

        if use_broken_axis:
            ax_bottom.set_xlabel(region_key.capitalize(), fontsize=9)
        else:
            ax.set_xlabel(region_key.capitalize(), fontsize=9)

        # ticks
        if use_broken_axis:
            plt.setp(ax_bottom.get_xticklabels(), rotation=45, ha='right', fontsize=8)
            plt.setp(ax_top.get_xticklabels(), rotation=45, ha='right', fontsize=8)
            ax_top.tick_params(axis='y', labelsize=8)
            ax_bottom.tick_params(axis='y', labelsize=8)
        else:
            plt.xticks(rotation=45, ha='right', fontsize=8)
            plt.yticks(fontsize=8)

        fig.tight_layout(rect=(0, 0, 0.7, 1))

        safe_cell_type = cell_type.replace(" ", "_").replace("-", "_")
        plot_filename = os.path.join(
            savedir,
            f"Fig2_suppl_{safe_cell_type}_proportions_boxplot-Breslow.pdf"
        )

        try:
            fig.savefig(plot_filename)
            print(f"  Plot for {cell_type} saved to: {plot_filename}")
        except Exception as e:
            print(f"  Error saving plot for {cell_type}: {e}")

        plt.close(fig)

In [67]:
proportion_results=pd.read_csv(r"C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\supplementary materials\CyCIF_data_for_plotting\Fig2C-NGFR-single-marker-merged-data_for-plotting.csv")

plot_proportions_by_celltype(
    df=proportion_results,
    region_key='ROI_major_category',
    celltype_key='NGFRpos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order  ,
    bottom_ylim=(0, 0.1),
    top_ylim=(0.7, 1.0),
    figsize_broken=(1.6, 2),
    broken_height_ratios=(1, 3)
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:249: UserWarning:

This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.



  Plot for NGFR+ saved to: celltype_proportion_plots\Fig2_suppl_NGFR+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:249: UserWarning:

This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.



  Plot for NGFR- saved to: celltype_proportion_plots\Fig2_suppl_NGFR__proportions_boxplot-Breslow.pdf


In [30]:
proportion_results=pd.read_csv(r"C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\supplementary materials\CyCIF_data_for_plotting\Fig2C-MART1-single-marker-merged-data_for-plotting.csv")

order=['normal','precursor','MIS','invasive']

custom_colors = {
    "normal":   "#4782b6",
    "precursor":"#278b44",
    "MIS":      "#ea9ec5",
    "invasive": "#f1634b"
}

plot_proportions_by_celltype(
    df=proportion_results,
    region_key='ROI_major_category',
    celltype_key='NGFRpos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order  ,
    bottom_ylim=(0, 0.1),
    top_ylim=(0.7, 1.0),
    figsize_broken=(2, 2),
    broken_height_ratios=(1, 3)
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2172942863.py:86: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for MART1+ saved to: celltype_proportion_plots\Fig2_suppl_MART1+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2172942863.py:86: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for MART1- saved to: celltype_proportion_plots\Fig2_suppl_MART1__proportions_boxplot-Breslow.pdf


#### SOX9

In [68]:
# final script used for Fig2C SOX9
def plot_proportions_by_celltype(
    df,
    region_key='ROI_major_category',
    celltype_key='lineage',
    sample_key='MEL_id',
    region_order=None,
    box_palette=None,
    savedir='celltype_proportion_plots',
    figsize=(3, 3)   # fixed figure size
):
    """
    Generates boxplots of proportions for each cell type, with:
      - Mann–Whitney U and Levene tests between region pairs (only p<=0.05 shown)
      - Samples with no ROI/celltype combinations filled with 0 for proportion/count/region_total.
      - Fixed figure size and fixed subplot margins so x-axis width is constant.
    """
    os.makedirs(savedir, exist_ok=True)
    sns.set_style("whitegrid")

    # ---------- EXPAND DF: per-sample, per-ROI × all cell types; fill missing with 0 ----------
    all_celltypes = df[celltype_key].unique()
    expanded_parts = []

    for sample_id, sample_df in df.groupby(sample_key):
        sample_rois = sample_df[region_key].dropna().unique()

        full_index = pd.MultiIndex.from_product(
            [sample_rois, all_celltypes],
            names=[region_key, celltype_key]
        )

        sample_expanded = (
            sample_df
            .set_index([region_key, celltype_key])
            .reindex(full_index)
            .reset_index()
        )

        sample_expanded[sample_key] = sample_id

        if 'proportion' in sample_expanded.columns:
            sample_expanded['proportion'] = sample_expanded['proportion'].fillna(0)
        if 'count' in sample_expanded.columns:
            sample_expanded['count'] = sample_expanded['count'].fillna(0).astype(int)
        if 'region_total' in sample_expanded.columns:
            sample_expanded['region_total'] = sample_expanded['region_total'].fillna(0).astype(int)

        expanded_parts.append(sample_expanded)

    df_expanded = pd.concat(expanded_parts, ignore_index=True)

    # ---------- PLOTTING ----------
    unique_celltypes = df_expanded[celltype_key].unique()
    print(f"\nGenerating {len(unique_celltypes)} boxplots (one for each cell type).")

    for cell_type in unique_celltypes:
        plot_df = df_expanded[df_expanded[celltype_key] == cell_type].copy()
        if plot_df.empty:
            print(f"Skipping plot for {cell_type}: No data available.")
            continue

        # fixed-size figure for every cell type
        fig, ax = plt.subplots(figsize=figsize)

        # fixed margins so the axes width is constant across plots
        # (adjust these numbers if you want more/less space for labels)
        fig.subplots_adjust(left=0.22, right=0.95, bottom=0.35, top=0.88)

        # ----- region order -----
        if region_order is None:
            this_region_order = sorted(plot_df[region_key].dropna().unique())
        else:
            this_region_order = region_order

        palette_to_use = box_palette if box_palette is not None else 'Set2'

        # ----- boxplot -----
        ax = sns.boxplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            palette=palette_to_use,
            width=0.5,
            showfliers=False,
            linewidth=0.5,
            ax=ax
        )

        ylims = ax.get_ylim()

        # ----- stripplot -----
        ax = sns.stripplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            size=1,
            jitter=True,
            dodge=False,
            edgecolor='black',
            linewidth=0.5,
            ax=ax
        )

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        # ----- add n above each box -----
        stats = (
            plot_df
            .groupby(region_key)['proportion']
            .agg(count='size', ymax='max')
            .reindex(this_region_order)
        )

        tick_locs = ax.get_xticks()
        tick_labels = [t.get_text() for t in ax.get_xticklabels()]
        x_pos_map = dict(zip(tick_labels, tick_locs))

        for region, row in stats.iterrows():
            if pd.isna(row['count']):
                continue
            n = int(row['count'])
            if region not in x_pos_map:
                continue
            x = x_pos_map[region]
            y_raw = (row['ymax'] * 1.05) if pd.notnull(row['ymax']) else 0.0
            y = min(y_raw, 0.98)

            ax.text(
                x,
                y,
                f"n={n}",
                ha="center",
                va="bottom",
                fontsize=7,
                color="black"
            )

        # ----- Mann–Whitney U and Levene tests between each pair of regions -----
        lines = []
        region_pairs = list(itertools.combinations(this_region_order, 2))

        for r1, r2 in region_pairs:
            g1 = plot_df.loc[plot_df[region_key] == r1, 'proportion'].dropna()
            g2 = plot_df.loc[plot_df[region_key] == r2, 'proportion'].dropna()

            if len(g1) < 2 or len(g2) < 2:
                continue

            stat_mw, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
            stat_lev, p_lev = levene(g1, g2, center='median')

            pair_header_added = False

            if p_mw <= 0.05:
                lines.append(f"{r1} vs {r2}:")
                lines.append(f"  MWU p={p_mw:.2e}")
                pair_header_added = True

            if p_lev <= 0.05:
                if not pair_header_added:
                    lines.append(f"{r1} vs {r2}:")
                lines.append(f"  Lev p={p_lev:.2e}")

        if lines:
            ax.text(
                0.72, 0.98,
                "\n".join(lines),
                transform=ax.transAxes,
                va='top',
                ha='left',
                fontsize=6,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8)
            )

        # ----- aesthetics -----
        ax.set_title(f"{cell_type}", fontsize=10)
        ax.set_xlabel(region_key.capitalize(), fontsize=9)
        ax.set_ylabel(f"Proportion of {cell_type}", fontsize=9)
        ax.set(ylim=ylims)
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.yticks(fontsize=8)

        safe_cell_type = cell_type.replace(" ", "_").replace("-", "_")
        plot_filename = os.path.join(
            savedir,
            f"Fig2_suppl_{safe_cell_type}_proportions_boxplot-Breslow.pdf"
        )

        try:
            fig.savefig(plot_filename)  # no bbox_inches='tight'
            print(f"  Plot for {cell_type} saved to: {plot_filename}")
        except Exception as e:
            print(f"  Error saving plot for {cell_type}: {e}")

        plt.close(fig)

In [69]:
proportion_results=pd.read_csv(r"C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\supplementary materials\CyCIF_data_for_plotting\Fig2C-SOX9-single-marker-merged-data_for-plotting.csv")


order=['normal','precursor','MIS','invasive']

custom_colors = {
    "normal":   "#4782b6",
    "precursor":"#278b44",
    "MIS":      "#ea9ec5",
    "invasive": "#f1634b"
}

plot_proportions_by_celltype(
    df=proportion_results,
    region_key='ROI_major_category',
    celltype_key='SOX9pos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
    #savedir=r'C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\3-Revision Nature Cancer March 2026\data\figure_panels'
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\1157819311.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for SOX9+ saved to: celltype_proportion_plots\Fig2_suppl_SOX9+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\1157819311.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for SOX9- saved to: celltype_proportion_plots\Fig2_suppl_SOX9__proportions_boxplot-Breslow.pdf


#### S100A1

In [70]:
# final script used for S100A1
def plot_proportions_by_celltype(
    df,
    region_key='ROI_major_category',
    celltype_key='lineage',
    sample_key='MEL_id',
    region_order=None,
    box_palette=None,
    savedir='celltype_proportion_plots',
    figsize=(3, 3)   # fixed figure size
):
    """
    Generates boxplots of proportions for each cell type, with:
      - Mann–Whitney U and Levene tests between region pairs (only p<=0.05 shown)
      - Samples with no ROI/celltype combinations filled with 0 for proportion/count/region_total.
      - Fixed figure size and fixed subplot margins so x-axis width is constant.
    """
    os.makedirs(savedir, exist_ok=True)
    sns.set_style("whitegrid")

    # ---------- EXPAND DF: per-sample, per-ROI × all cell types; fill missing with 0 ----------
    all_celltypes = df[celltype_key].unique()
    expanded_parts = []

    for sample_id, sample_df in df.groupby(sample_key):
        sample_rois = sample_df[region_key].dropna().unique()

        full_index = pd.MultiIndex.from_product(
            [sample_rois, all_celltypes],
            names=[region_key, celltype_key]
        )

        sample_expanded = (
            sample_df
            .set_index([region_key, celltype_key])
            .reindex(full_index)
            .reset_index()
        )

        sample_expanded[sample_key] = sample_id

        if 'proportion' in sample_expanded.columns:
            sample_expanded['proportion'] = sample_expanded['proportion'].fillna(0)
        if 'count' in sample_expanded.columns:
            sample_expanded['count'] = sample_expanded['count'].fillna(0).astype(int)
        if 'region_total' in sample_expanded.columns:
            sample_expanded['region_total'] = sample_expanded['region_total'].fillna(0).astype(int)

        expanded_parts.append(sample_expanded)

    df_expanded = pd.concat(expanded_parts, ignore_index=True)

    # ---------- PLOTTING ----------
    unique_celltypes = df_expanded[celltype_key].unique()
    print(f"\nGenerating {len(unique_celltypes)} boxplots (one for each cell type).")

    for cell_type in unique_celltypes:
        plot_df = df_expanded[df_expanded[celltype_key] == cell_type].copy()
        if plot_df.empty:
            print(f"Skipping plot for {cell_type}: No data available.")
            continue

        # fixed-size figure for every cell type
        fig, ax = plt.subplots(figsize=figsize)

        # fixed margins so the axes width is constant across plots
        # (adjust these numbers if you want more/less space for labels)
        fig.subplots_adjust(left=0.22, right=0.95, bottom=0.35, top=0.88)

        # ----- region order -----
        if region_order is None:
            this_region_order = sorted(plot_df[region_key].dropna().unique())
        else:
            this_region_order = region_order

        palette_to_use = box_palette if box_palette is not None else 'Set2'

        # ----- boxplot -----
        ax = sns.boxplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            palette=palette_to_use,
            width=0.5,
            showfliers=False,
            linewidth=0.5,
            ax=ax
        )

        ylims = ax.get_ylim()

        # ----- stripplot -----
        ax = sns.stripplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            size=1,
            jitter=True,
            dodge=False,
            edgecolor='black',
            linewidth=0.5,
            ax=ax
        )

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        # ----- add n above each box -----
        stats = (
            plot_df
            .groupby(region_key)['proportion']
            .agg(count='size', ymax='max')
            .reindex(this_region_order)
        )

        tick_locs = ax.get_xticks()
        tick_labels = [t.get_text() for t in ax.get_xticklabels()]
        x_pos_map = dict(zip(tick_labels, tick_locs))

        for region, row in stats.iterrows():
            if pd.isna(row['count']):
                continue
            n = int(row['count'])
            if region not in x_pos_map:
                continue
            x = x_pos_map[region]
            y_raw = (row['ymax'] * 1.05) if pd.notnull(row['ymax']) else 0.0
            y = min(y_raw, 0.98)

            ax.text(
                x,
                y,
                f"n={n}",
                ha="center",
                va="bottom",
                fontsize=7,
                color="black"
            )

        # ----- Mann–Whitney U and Levene tests between each pair of regions -----
        lines = []
        region_pairs = list(itertools.combinations(this_region_order, 2))

        for r1, r2 in region_pairs:
            g1 = plot_df.loc[plot_df[region_key] == r1, 'proportion'].dropna()
            g2 = plot_df.loc[plot_df[region_key] == r2, 'proportion'].dropna()

            if len(g1) < 2 or len(g2) < 2:
                continue

            stat_mw, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
            stat_lev, p_lev = levene(g1, g2, center='median')

            pair_header_added = False

            if p_mw <= 0.05:
                lines.append(f"{r1} vs {r2}:")
                lines.append(f"  MWU p={p_mw:.2e}")
                pair_header_added = True

            if p_lev <= 0.05:
                if not pair_header_added:
                    lines.append(f"{r1} vs {r2}:")
                lines.append(f"  Lev p={p_lev:.2e}")

        if lines:
            ax.text(
                0.72, 0.98,
                "\n".join(lines),
                transform=ax.transAxes,
                va='top',
                ha='left',
                fontsize=6,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8)
            )

        # ----- aesthetics -----
        ax.set_title(f"{cell_type}", fontsize=10)
        ax.set_xlabel(region_key.capitalize(), fontsize=9)
        ax.set_ylabel(f"Proportion of {cell_type}", fontsize=9)
        ax.set(ylim=ylims)
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.yticks(fontsize=8)

        safe_cell_type = cell_type.replace(" ", "_").replace("-", "_")
        plot_filename = os.path.join(
            savedir,
            f"Fig2_suppl_{safe_cell_type}_proportions_boxplot-Breslow.pdf"
        )

        try:
            fig.savefig(plot_filename)  # no bbox_inches='tight'
            print(f"  Plot for {cell_type} saved to: {plot_filename}")
        except Exception as e:
            print(f"  Error saving plot for {cell_type}: {e}")

        plt.close(fig)

In [71]:
proportion_results=pd.read_csv(r"C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\supplementary materials\CyCIF_data_for_plotting\Fig2C-S100A1-single-marker-merged-data_for-plotting.csv")

order=['normal','precursor','MIS','invasive']

custom_colors = {
    "normal":   "#4782b6",
    "precursor":"#278b44",
    "MIS":      "#ea9ec5",
    "invasive": "#f1634b"
}

plot_proportions_by_celltype(
    df=proportion_results,
    region_key='ROI_major_category',
    celltype_key='S100A1pos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\438704524.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for S100A1+ saved to: celltype_proportion_plots\Fig2_suppl_S100A1+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\438704524.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for S100A1- saved to: celltype_proportion_plots\Fig2_suppl_S100A1__proportions_boxplot-Breslow.pdf


#### PRAME

In [72]:
proportion_results=pd.read_csv(r"C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\supplementary materials\CyCIF_data_for_plotting\Fig2C-PRAME-single-marker-merged-data_for-plotting.csv")

order=['normal','precursor','MIS','invasive']

custom_colors = {
    "normal":   "#4782b6",
    "precursor":"#278b44",
    "MIS":      "#ea9ec5",
    "invasive": "#f1634b"
}

plot_proportions_by_celltype(
    df=proportion_results,
    region_key='ROI_major_category',
    celltype_key='PRAMEpos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\438704524.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for PRAME+ saved to: celltype_proportion_plots\Fig2_suppl_PRAME+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\438704524.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for PRAME- saved to: celltype_proportion_plots\Fig2_suppl_PRAME__proportions_boxplot-Breslow.pdf


### Extended Fig 3D,E (single-marker Breslow plots)

In [73]:

def calculate_regional_cell_proportions(df, sample_key='MELid', region_key='ROI_major_category', celltype_key='lineage'):
    """
    Calculates the proportion of each cell type within each unique region, 
    separately for each sample.

    Args:
        df (pd.DataFrame): The DataFrame containing single-cell data.
        sample_key (str): The column name representing the sample ID (e.g., 'ID').
        region_key (str): The column name representing the region (e.g., 'region').
        celltype_key (str): The column name representing the cell type (e.g., 'celltype').

    Returns:
        pd.DataFrame: A DataFrame with proportions, counts, and region totals 
                      for every cell type/region/sample combination.
    """
    results = []
    
    required_keys = [sample_key, region_key, celltype_key]
    if not all(key in df.columns for key in required_keys):
        missing_keys = [key for key in required_keys if key not in df.columns]
        print(f"Error: DataFrame is missing required columns: {missing_keys}")
        return pd.DataFrame()

    unique_samples = df[sample_key].unique()
    print(f"\nCalculating cell type proportions for {len(unique_samples)} unique samples.")

    # Loop through each sample
    for sample_id in unique_samples:
        print(f"Processing sample: {sample_id}...")
        
        # Subset the data for the current sample
        sample_df = df[df[sample_key] == sample_id].copy()

        # 1. Calculate cell type counts per region (Count of CellType A in Region X)
        # group by both region and cell type
        celltype_counts = sample_df.groupby([region_key, celltype_key]).size().reset_index(name='count')

        # 2. Calculate total cell count per region (Total cells in Region X)
        region_totals = sample_df.groupby(region_key).size().reset_index(name='region_total')
        
        # 3. Merge counts with totals on the region key
        merged_counts = pd.merge(celltype_counts, region_totals, on=region_key)

        # 4. Calculate proportion: (Count of A in X) / (Total cells in X)
        merged_counts['proportion'] = merged_counts['count'] / merged_counts['region_total']
        
        # 5. Add sample ID, column order
        merged_counts[sample_key] = sample_id
        final_cols = [sample_key, region_key, celltype_key, 'proportion', 'count', 'region_total']
        
        results.append(merged_counts[final_cols])

    # Combine results
    final_proportions_df = pd.concat(results, ignore_index=True)
    return final_proportions_df



In [83]:
merged_data=pd.read_csv(r"C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\supplementary materials\CyCIF_data_for_plotting\Extended-Fig3D-3E-single-marker-merged-data_for-plotting.csv")

C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2450383607.py:1: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.



In [84]:
proportion_results=calculate_regional_cell_proportions(df=merged_data, sample_key='MELid', region_key='Breslow_cat', celltype_key='MART1pos')


Calculating cell type proportions for 37 unique samples.
Processing sample: MEL14...
Processing sample: MEL16...
Processing sample: MEL18...
Processing sample: MEL19...
Processing sample: MEL25...
Processing sample: MEL26...
Processing sample: MEL39...
Processing sample: MEL40...
Processing sample: MEL44...
Processing sample: MEL53...
Processing sample: MEL60...
Processing sample: MEL66...
Processing sample: MEL68_B1...
Processing sample: MEL69...
Processing sample: MEL21...
Processing sample: MEL31...
Processing sample: MEL34...
Processing sample: MEL35...
Processing sample: MEL36_A2...
Processing sample: MEL51...
Processing sample: MEL54...
Processing sample: MEL55...
Processing sample: MEL59...
Processing sample: MEL50...
Processing sample: MEL73...
Processing sample: MEL78...
Processing sample: MEL70...
Processing sample: MEL72...
Processing sample: MEL74...
Processing sample: MEL76...
Processing sample: MEL84...
Processing sample: MEL85...
Processing sample: MEL81...
Processing s

In [85]:
merged_data["Breslow_cat"].value_counts()

Breslow_cat
>4     116430
1–2     71837
<1      70455
2–4     50559
Name: count, dtype: int64

In [86]:
# final script used for Fig2C KI67
def plot_proportions_by_celltype(
    df,
    region_key='ROI_major_category',
    celltype_key='lineage',
    sample_key='MEL_id',
    region_order=None,
    box_palette=None,
    savedir='celltype_proportion_plots_Breslow',
    figsize=(3, 3)   # fixed figure size
):
    """
    Generates boxplots of proportions for each cell type, with:
      - Mann–Whitney U and Levene tests between region pairs (only p<=0.05 shown)
      - Samples with no ROI/celltype combinations filled with 0 for proportion/count/region_total.
      - Fixed figure size and fixed subplot margins so x-axis width is constant.
    """
    os.makedirs(savedir, exist_ok=True)
    sns.set_style("whitegrid")

    # ---------- EXPAND DF: per-sample, per-ROI × all cell types; fill missing with 0 ----------
    all_celltypes = df[celltype_key].unique()
    expanded_parts = []

    for sample_id, sample_df in df.groupby(sample_key):
        sample_rois = sample_df[region_key].dropna().unique()

        full_index = pd.MultiIndex.from_product(
            [sample_rois, all_celltypes],
            names=[region_key, celltype_key]
        )

        sample_expanded = (
            sample_df
            .set_index([region_key, celltype_key])
            .reindex(full_index)
            .reset_index()
        )

        sample_expanded[sample_key] = sample_id

        if 'proportion' in sample_expanded.columns:
            sample_expanded['proportion'] = sample_expanded['proportion'].fillna(0)
        if 'count' in sample_expanded.columns:
            sample_expanded['count'] = sample_expanded['count'].fillna(0).astype(int)
        if 'region_total' in sample_expanded.columns:
            sample_expanded['region_total'] = sample_expanded['region_total'].fillna(0).astype(int)

        expanded_parts.append(sample_expanded)

    df_expanded = pd.concat(expanded_parts, ignore_index=True)

    # ---------- PLOTTING ----------
    unique_celltypes = df_expanded[celltype_key].unique()
    print(f"\nGenerating {len(unique_celltypes)} boxplots (one for each cell type).")

    for cell_type in unique_celltypes:
        plot_df = df_expanded[df_expanded[celltype_key] == cell_type].copy()
        if plot_df.empty:
            print(f"Skipping plot for {cell_type}: No data available.")
            continue

        # fixed-size figure for every cell type
        fig, ax = plt.subplots(figsize=figsize)

        # fixed margins so the axes width is constant across plots
        # (adjust these numbers if you want more/less space for labels)
        fig.subplots_adjust(left=0.22, right=0.95, bottom=0.35, top=0.88)

        # ----- region order -----
        if region_order is None:
            this_region_order = sorted(plot_df[region_key].dropna().unique())
        else:
            this_region_order = region_order

        palette_to_use = box_palette if box_palette is not None else 'Set2'

        # ----- boxplot -----
        ax = sns.boxplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            palette=palette_to_use,
            width=0.5,
            showfliers=False,
            linewidth=0.5,
            ax=ax
        )

        ylims = ax.get_ylim()

        # ----- stripplot -----
        ax = sns.stripplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            size=1,
            jitter=True,
            dodge=False,
            edgecolor='black',
            linewidth=0.5,
            ax=ax
        )

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        # ----- add n above each box -----
        stats = (
            plot_df
            .groupby(region_key)['proportion']
            .agg(count='size', ymax='max')
            .reindex(this_region_order)
        )

        tick_locs = ax.get_xticks()
        tick_labels = [t.get_text() for t in ax.get_xticklabels()]
        x_pos_map = dict(zip(tick_labels, tick_locs))

        for region, row in stats.iterrows():
            if pd.isna(row['count']):
                continue
            n = int(row['count'])
            if region not in x_pos_map:
                continue
            x = x_pos_map[region]
            y_raw = (row['ymax'] * 1.05) if pd.notnull(row['ymax']) else 0.0
            y = min(y_raw, 0.98)

            ax.text(
                x,
                y,
                f"n={n}",
                ha="center",
                va="bottom",
                fontsize=7,
                color="black"
            )

        # ----- Mann–Whitney U and Levene tests between each pair of regions -----
        lines = []
        region_pairs = list(itertools.combinations(this_region_order, 2))

        for r1, r2 in region_pairs:
            g1 = plot_df.loc[plot_df[region_key] == r1, 'proportion'].dropna()
            g2 = plot_df.loc[plot_df[region_key] == r2, 'proportion'].dropna()

            if len(g1) < 2 or len(g2) < 2:
                continue

            stat_mw, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
            stat_lev, p_lev = levene(g1, g2, center='median')

            pair_header_added = False

            if p_mw <= 0.05:
                lines.append(f"{r1} vs {r2}:")
                lines.append(f"  MWU p={p_mw:.2e}")
                pair_header_added = True

            if p_lev <= 0.05:
                if not pair_header_added:
                    lines.append(f"{r1} vs {r2}:")
                lines.append(f"  Lev p={p_lev:.2e}")

        if lines:
            ax.text(
                0.72, 0.98,
                "\n".join(lines),
                transform=ax.transAxes,
                va='top',
                ha='left',
                fontsize=6,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8)
            )

        # ----- aesthetics -----
        ax.set_title(f"{cell_type}", fontsize=10)
        ax.set_xlabel(region_key.capitalize(), fontsize=9)
        ax.set_ylabel(f"Proportion of {cell_type}", fontsize=9)
        ax.set(ylim=ylims)
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.yticks(fontsize=8)

        safe_cell_type = cell_type.replace(" ", "_").replace("-", "_")
        plot_filename = os.path.join(
            savedir,
            f"Fig2_suppl_{safe_cell_type}_proportions_boxplot-Breslow.pdf"
        )

        try:
            fig.savefig(plot_filename)  # no bbox_inches='tight'
            print(f"  Plot for {cell_type} saved to: {plot_filename}")
        except Exception as e:
            print(f"  Error saving plot for {cell_type}: {e}")

        plt.close(fig)

In [87]:
order=['<1','1–2','2–4','>4']

custom_colors = {
    "<1":   "#dd88b8",
    "1–2":"#ddd524",
    "2–4":      "#4993c5",
    ">4": "#40b774"
}

plot_proportions_by_celltype(
    df=proportion_results,
    region_key='Breslow_cat',
    celltype_key='MART1pos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
    #savedir=r'C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\3-Revision Nature Cancer March 2026\data\figure_panels'
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for MART1+ saved to: celltype_proportion_plots_Breslow\Fig2_suppl_MART1+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for MART1- saved to: celltype_proportion_plots_Breslow\Fig2_suppl_MART1__proportions_boxplot-Breslow.pdf


In [88]:
proportion_results_NGFR=calculate_regional_cell_proportions(df=merged_data, sample_key='MELid', region_key='Breslow_cat', celltype_key='NGFRpos')


Calculating cell type proportions for 37 unique samples.
Processing sample: MEL14...
Processing sample: MEL16...
Processing sample: MEL18...
Processing sample: MEL19...
Processing sample: MEL25...
Processing sample: MEL26...
Processing sample: MEL39...
Processing sample: MEL40...
Processing sample: MEL44...
Processing sample: MEL53...
Processing sample: MEL60...
Processing sample: MEL66...
Processing sample: MEL68_B1...
Processing sample: MEL69...
Processing sample: MEL21...
Processing sample: MEL31...
Processing sample: MEL34...
Processing sample: MEL35...
Processing sample: MEL36_A2...
Processing sample: MEL51...
Processing sample: MEL54...
Processing sample: MEL55...
Processing sample: MEL59...
Processing sample: MEL50...
Processing sample: MEL73...
Processing sample: MEL78...
Processing sample: MEL70...
Processing sample: MEL72...
Processing sample: MEL74...
Processing sample: MEL76...
Processing sample: MEL84...
Processing sample: MEL85...
Processing sample: MEL81...
Processing s

In [89]:
order=['<1','1–2','2–4','>4']

custom_colors = {
    "<1":   "#dd88b8",
    "1–2":"#ddd524",
    "2–4":      "#4993c5",
    ">4": "#40b774"
}

plot_proportions_by_celltype(
    df=proportion_results_NGFR,
    region_key='Breslow_cat',
    celltype_key='NGFRpos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
    #savedir=r'C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\3-Revision Nature Cancer March 2026\data\figure_panels'
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for NGFR+ saved to: celltype_proportion_plots_Breslow\Fig2_suppl_NGFR+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for NGFR- saved to: celltype_proportion_plots_Breslow\Fig2_suppl_NGFR__proportions_boxplot-Breslow.pdf


In [91]:
proportion_results_KI67=calculate_regional_cell_proportions(df=merged_data, sample_key='MELid', region_key='Breslow_cat', celltype_key='Ki67pos')


Calculating cell type proportions for 37 unique samples.
Processing sample: MEL14...
Processing sample: MEL16...
Processing sample: MEL18...
Processing sample: MEL19...
Processing sample: MEL25...
Processing sample: MEL26...
Processing sample: MEL39...
Processing sample: MEL40...
Processing sample: MEL44...
Processing sample: MEL53...
Processing sample: MEL60...
Processing sample: MEL66...
Processing sample: MEL68_B1...
Processing sample: MEL69...
Processing sample: MEL21...
Processing sample: MEL31...
Processing sample: MEL34...
Processing sample: MEL35...
Processing sample: MEL36_A2...
Processing sample: MEL51...
Processing sample: MEL54...
Processing sample: MEL55...
Processing sample: MEL59...
Processing sample: MEL50...
Processing sample: MEL73...
Processing sample: MEL78...
Processing sample: MEL70...
Processing sample: MEL72...
Processing sample: MEL74...
Processing sample: MEL76...
Processing sample: MEL84...
Processing sample: MEL85...
Processing sample: MEL81...
Processing s

In [93]:
order=['<1','1–2','2–4','>4']

custom_colors = {
    "<1":   "#dd88b8",
    "1–2":"#ddd524",
    "2–4":      "#4993c5",
    ">4": "#40b774"
}

plot_proportions_by_celltype(
    df=proportion_results_KI67,
    region_key='Breslow_cat',
    celltype_key='Ki67pos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
    #savedir=r'C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\3-Revision Nature Cancer March 2026\data\figure_panels'
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for Ki67+ saved to: celltype_proportion_plots_Breslow\Fig2_suppl_Ki67+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for Ki67- saved to: celltype_proportion_plots_Breslow\Fig2_suppl_Ki67__proportions_boxplot-Breslow.pdf


In [94]:
proportion_results_SOX9=calculate_regional_cell_proportions(df=merged_data, sample_key='MELid', region_key='Breslow_cat', celltype_key='SOX9pos')


Calculating cell type proportions for 37 unique samples.
Processing sample: MEL14...
Processing sample: MEL16...
Processing sample: MEL18...
Processing sample: MEL19...
Processing sample: MEL25...
Processing sample: MEL26...
Processing sample: MEL39...
Processing sample: MEL40...
Processing sample: MEL44...
Processing sample: MEL53...
Processing sample: MEL60...
Processing sample: MEL66...
Processing sample: MEL68_B1...
Processing sample: MEL69...
Processing sample: MEL21...
Processing sample: MEL31...
Processing sample: MEL34...
Processing sample: MEL35...
Processing sample: MEL36_A2...
Processing sample: MEL51...
Processing sample: MEL54...
Processing sample: MEL55...
Processing sample: MEL59...
Processing sample: MEL50...
Processing sample: MEL73...
Processing sample: MEL78...
Processing sample: MEL70...
Processing sample: MEL72...
Processing sample: MEL74...
Processing sample: MEL76...
Processing sample: MEL84...
Processing sample: MEL85...
Processing sample: MEL81...
Processing s

In [95]:
order=['<1','1–2','2–4','>4']

custom_colors = {
    "<1":   "#dd88b8",
    "1–2":"#ddd524",
    "2–4":      "#4993c5",
    ">4": "#40b774"
}

plot_proportions_by_celltype(
    df=proportion_results_SOX9,
    region_key='Breslow_cat',
    celltype_key='SOX9pos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
    #savedir=r'C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\3-Revision Nature Cancer March 2026\data\figure_panels'
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for SOX9+ saved to: celltype_proportion_plots_Breslow\Fig2_suppl_SOX9+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for SOX9- saved to: celltype_proportion_plots_Breslow\Fig2_suppl_SOX9__proportions_boxplot-Breslow.pdf


In [96]:
proportion_results_PRAME=calculate_regional_cell_proportions(df=merged_data, sample_key='MELid', region_key='Breslow_cat', celltype_key='PRAMEpos')


Calculating cell type proportions for 37 unique samples.
Processing sample: MEL14...
Processing sample: MEL16...
Processing sample: MEL18...
Processing sample: MEL19...
Processing sample: MEL25...
Processing sample: MEL26...
Processing sample: MEL39...
Processing sample: MEL40...
Processing sample: MEL44...
Processing sample: MEL53...
Processing sample: MEL60...
Processing sample: MEL66...
Processing sample: MEL68_B1...
Processing sample: MEL69...
Processing sample: MEL21...
Processing sample: MEL31...
Processing sample: MEL34...
Processing sample: MEL35...
Processing sample: MEL36_A2...
Processing sample: MEL51...
Processing sample: MEL54...
Processing sample: MEL55...
Processing sample: MEL59...
Processing sample: MEL50...
Processing sample: MEL73...
Processing sample: MEL78...
Processing sample: MEL70...
Processing sample: MEL72...
Processing sample: MEL74...
Processing sample: MEL76...
Processing sample: MEL84...
Processing sample: MEL85...
Processing sample: MEL81...
Processing s

In [97]:
order=['<1','1–2','2–4','>4']

custom_colors = {
    "<1":   "#dd88b8",
    "1–2":"#ddd524",
    "2–4":      "#4993c5",
    ">4": "#40b774"
}

plot_proportions_by_celltype(
    df=proportion_results_PRAME,
    region_key='Breslow_cat',
    celltype_key='PRAMEpos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
    #savedir=r'C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\3-Revision Nature Cancer March 2026\data\figure_panels'
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for PRAME+ saved to: celltype_proportion_plots_Breslow\Fig2_suppl_PRAME+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for PRAME- saved to: celltype_proportion_plots_Breslow\Fig2_suppl_PRAME__proportions_boxplot-Breslow.pdf


In [98]:
proportion_results_S100A1=calculate_regional_cell_proportions(df=merged_data, sample_key='MELid', region_key='Breslow_cat', celltype_key='S100A1pos')


Calculating cell type proportions for 37 unique samples.
Processing sample: MEL14...
Processing sample: MEL16...
Processing sample: MEL18...
Processing sample: MEL19...
Processing sample: MEL25...
Processing sample: MEL26...
Processing sample: MEL39...
Processing sample: MEL40...
Processing sample: MEL44...
Processing sample: MEL53...
Processing sample: MEL60...
Processing sample: MEL66...
Processing sample: MEL68_B1...
Processing sample: MEL69...
Processing sample: MEL21...
Processing sample: MEL31...
Processing sample: MEL34...
Processing sample: MEL35...
Processing sample: MEL36_A2...
Processing sample: MEL51...
Processing sample: MEL54...
Processing sample: MEL55...
Processing sample: MEL59...
Processing sample: MEL50...
Processing sample: MEL73...
Processing sample: MEL78...
Processing sample: MEL70...
Processing sample: MEL72...
Processing sample: MEL74...
Processing sample: MEL76...
Processing sample: MEL84...
Processing sample: MEL85...
Processing sample: MEL81...
Processing s

In [100]:
order=['<1','1–2','2–4','>4']

custom_colors = {
    "<1":   "#dd88b8",
    "1–2":"#ddd524",
    "2–4":      "#4993c5",
    ">4": "#40b774"
}

plot_proportions_by_celltype(
    df=proportion_results_S100A1,
    region_key='Breslow_cat',
    celltype_key='S100A1pos',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order,
    figsize=(1.6, 3)
    #savedir=r'C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\3-Revision Nature Cancer March 2026\data\figure_panels'
)


Generating 2 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for S100A1+ saved to: celltype_proportion_plots_Breslow\Fig2_suppl_S100A1+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\2059927944.py:79: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for S100A1- saved to: celltype_proportion_plots_Breslow\Fig2_suppl_S100A1__proportions_boxplot-Breslow.pdf


### Multimarker tumor state plots (Fig 2, Ext Fig 3C)

In [101]:
data=pd.read_csv(r"C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\supplementary materials\CyCIF_data_for_plotting\Fig2A-multimarker-tumor-state-merged-data_for-plotting.csv")

In [102]:
data

,Unnamed: 0,ROI_major_category,lineage,MELid,proportion,count,region_total
0,0,MIS,SOX10+SOX9+NGFR-MART1+,MEL14,0.057387,148,2579
1,1,MIS,SOX10+SOX9-NGFR-MART1+,MEL14,0.861187,2221,2579
2,2,MIS,SOX10+SOX9-NGFR-MART1-,MEL14,0.079488,205,2579
3,3,MIS,SOX10+SOX9+NGFR-MART1-,MEL14,0.000000,0,0
4,4,MIS,SOX10+SOX9-NGFR+MART1-,MEL14,0.000000,0,0
...,...,...,...,...,...,...,...
451,451,invasive,SOX10+SOX9-NGFR-MART1+,MEL86,0.022840,138,6042
452,452,invasive,SOX10+SOX9-NGFR-MART1-,MEL86,0.001821,11,6042
453,453,invasive,SOX10+SOX9+NGFR-MART1-,MEL86,0.002152,13,6042
454,454,invasive,SOX10+SOX9-NGFR+MART1-,MEL86,0.008110,49,6042


In [103]:
def plot_proportions_by_celltype(
    df,
    region_key='ROI_major_category',
    celltype_key='lineage',
    sample_key='MEL_id',
    region_order=None,
    box_palette=None,
    savedir='celltype_proportion_plots',
    figsize=(3, 3)   # fixed figure size
):
    """
    Generates boxplots of proportions for each cell type, with:
      - Mann–Whitney U and Levene tests between region pairs (only p<=0.05 shown)
      - Samples with no ROI/celltype combinations filled with 0 for proportion/count/region_total.
      - Fixed figure size and fixed subplot margins so x-axis width is constant.
    """
    os.makedirs(savedir, exist_ok=True)
    sns.set_style("whitegrid")

    # ---------- EXPAND DF: per-sample, per-ROI × all cell types; fill missing with 0 ----------
    all_celltypes = df[celltype_key].unique()
    expanded_parts = []

    for sample_id, sample_df in df.groupby(sample_key):
        sample_rois = sample_df[region_key].dropna().unique()

        full_index = pd.MultiIndex.from_product(
            [sample_rois, all_celltypes],
            names=[region_key, celltype_key]
        )

        sample_expanded = (
            sample_df
            .set_index([region_key, celltype_key])
            .reindex(full_index)
            .reset_index()
        )

        sample_expanded[sample_key] = sample_id

        if 'proportion' in sample_expanded.columns:
            sample_expanded['proportion'] = sample_expanded['proportion'].fillna(0)
        if 'count' in sample_expanded.columns:
            sample_expanded['count'] = sample_expanded['count'].fillna(0).astype(int)
        if 'region_total' in sample_expanded.columns:
            sample_expanded['region_total'] = sample_expanded['region_total'].fillna(0).astype(int)

        expanded_parts.append(sample_expanded)

    df_expanded = pd.concat(expanded_parts, ignore_index=True)

    # ---------- PLOTTING ----------
    unique_celltypes = df_expanded[celltype_key].unique()
    print(f"\nGenerating {len(unique_celltypes)} boxplots (one for each cell type).")

    for cell_type in unique_celltypes:
        plot_df = df_expanded[df_expanded[celltype_key] == cell_type].copy()
        if plot_df.empty:
            print(f"Skipping plot for {cell_type}: No data available.")
            continue

        # fixed-size figure for every cell type
        fig, ax = plt.subplots(figsize=figsize)

        # fixed margins so the axes width is constant across plots
        # (adjust these numbers if you want more/less space for labels)
        fig.subplots_adjust(left=0.22, right=0.95, bottom=0.35, top=0.88)

        # ----- region order -----
        if region_order is None:
            this_region_order = sorted(plot_df[region_key].dropna().unique())
        else:
            this_region_order = region_order

        palette_to_use = box_palette if box_palette is not None else 'Set2'

        # ----- boxplot -----
        ax = sns.boxplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            palette=palette_to_use,
            width=0.5,
            showfliers=False,
            linewidth=0.5,
            ax=ax
        )

        ylims = ax.get_ylim()

        # ----- stripplot -----
        ax = sns.stripplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            size=1,
            jitter=True,
            dodge=False,
            edgecolor='black',
            linewidth=0.5,
            ax=ax
        )

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        # ----- add n above each box -----
        stats = (
            plot_df
            .groupby(region_key)['proportion']
            .agg(count='size', ymax='max')
            .reindex(this_region_order)
        )

        tick_locs = ax.get_xticks()
        tick_labels = [t.get_text() for t in ax.get_xticklabels()]
        x_pos_map = dict(zip(tick_labels, tick_locs))

        for region, row in stats.iterrows():
            if pd.isna(row['count']):
                continue
            n = int(row['count'])
            if region not in x_pos_map:
                continue
            x = x_pos_map[region]
            y_raw = (row['ymax'] * 1.05) if pd.notnull(row['ymax']) else 0.0
            y = min(y_raw, 0.98)

            ax.text(
                x,
                y,
                f"n={n}",
                ha="center",
                va="bottom",
                fontsize=7,
                color="black"
            )

        # ----- Mann–Whitney U and Levene tests between each pair of regions -----
        lines = []
        region_pairs = list(itertools.combinations(this_region_order, 2))

        for r1, r2 in region_pairs:
            g1 = plot_df.loc[plot_df[region_key] == r1, 'proportion'].dropna()
            g2 = plot_df.loc[plot_df[region_key] == r2, 'proportion'].dropna()

            if len(g1) < 2 or len(g2) < 2:
                continue

            stat_mw, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
            stat_lev, p_lev = levene(g1, g2, center='median')

            pair_header_added = False

            if p_mw <= 0.05:
                lines.append(f"{r1} vs {r2}:")
                lines.append(f"  MWU p={p_mw:.2e}")
                pair_header_added = True

            if p_lev <= 0.05:
                if not pair_header_added:
                    lines.append(f"{r1} vs {r2}:")
                lines.append(f"  Lev p={p_lev:.2e}")

        if lines:
            ax.text(
                0.72, 0.98,
                "\n".join(lines),
                transform=ax.transAxes,
                va='top',
                ha='left',
                fontsize=6,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8)
            )

        # ----- aesthetics -----
        ax.set_title(f"{cell_type}", fontsize=10)
        ax.set_xlabel(region_key.capitalize(), fontsize=9)
        ax.set_ylabel(f"Proportion of {cell_type}", fontsize=9)
        ax.set(ylim=ylims)
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.yticks(fontsize=8)

        safe_cell_type = cell_type.replace(" ", "_").replace("-", "_")
        plot_filename = os.path.join(
            savedir,
            f"Fig2_suppl_{safe_cell_type}_proportions_boxplot-Breslow.pdf"
        )

        try:
            fig.savefig(plot_filename)  # no bbox_inches='tight'
            print(f"  Plot for {cell_type} saved to: {plot_filename}")
        except Exception as e:
            print(f"  Error saving plot for {cell_type}: {e}")

        plt.close(fig)

In [104]:
order=['normal','precursor','MIS','invasive']

custom_colors = {
    "normal":   "#4782b6",
    "precursor":"#278b44",
    "MIS":      "#ea9ec5",
    "invasive": "#f1634b"
}

plot_proportions_by_celltype(
        df=data, 
        region_key='ROI_major_category', 
        celltype_key='lineage',
        sample_key='MELid',
        box_palette=custom_colors,
        region_order=order,
        figsize=(1.6, 3)
        )
        #savedir=r'C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\3-Revision Nature Cancer March 2026\data\figure_panels')



Generating 6 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\3704007226.py:78: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for SOX10+SOX9+NGFR-MART1+ saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9+NGFR_MART1+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\3704007226.py:78: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for SOX10+SOX9-NGFR-MART1+ saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9_NGFR_MART1+_proportions_boxplot-Breslow.pdf


...

  Plot for SOX10+SOX9+NGFR-MART1- saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9+NGFR_MART1__proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\3704007226.py:78: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\anaconda3\envs\scimap_new\lib\site-packages\scipy\stats\_morestats.py:3310: RuntimeWarning:

invalid value encountered in scalar divide



  Plot for SOX10+SOX9-NGFR+MART1- saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9_NGFR+MART1__proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\3704007226.py:78: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\anaconda3\envs\scimap_new\lib\site-packages\scipy\stats\_morestats.py:3310: RuntimeWarning:

invalid value encountered in scalar divide



  Plot for SOX10-SOX9+NGFR-MART1- saved to: celltype_proportion_plots\Fig2_suppl_SOX10_SOX9+NGFR_MART1__proportions_boxplot-Breslow.pdf


### plotting for SOX10-SOX9+NGFR-MART1-

In [105]:
def plot_proportions_by_celltype(
    df,
    region_key='ROI_major_category',
    celltype_key='lineage',
    sample_key='MEL_id',
    region_order=None,
    box_palette=None,
    savedir='celltype_proportion_plots',
    figsize=(3, 3)   # fixed figure size
):
    """
    Generates boxplots of proportions for each cell type, with:
      - Mann–Whitney U and Levene tests between region pairs (only p<=0.05 shown)
      - Samples with no ROI/celltype combinations filled with 0 for proportion/count/region_total.
      - Fixed figure size and fixed subplot margins so x-axis width is constant.
    """
    os.makedirs(savedir, exist_ok=True)
    sns.set_style("whitegrid")

    # ---------- EXPAND DF: per-sample, per-ROI × all cell types; fill missing with 0 ----------
    all_celltypes = df[celltype_key].unique()
    expanded_parts = []

    for sample_id, sample_df in df.groupby(sample_key):
        sample_rois = sample_df[region_key].dropna().unique()

        full_index = pd.MultiIndex.from_product(
            [sample_rois, all_celltypes],
            names=[region_key, celltype_key]
        )

        sample_expanded = (
            sample_df
            .set_index([region_key, celltype_key])
            .reindex(full_index)
            .reset_index()
        )

        sample_expanded[sample_key] = sample_id

        if 'proportion' in sample_expanded.columns:
            sample_expanded['proportion'] = sample_expanded['proportion'].fillna(0)
        if 'count' in sample_expanded.columns:
            sample_expanded['count'] = sample_expanded['count'].fillna(0).astype(int)
        if 'region_total' in sample_expanded.columns:
            sample_expanded['region_total'] = sample_expanded['region_total'].fillna(0).astype(int)

        expanded_parts.append(sample_expanded)

    df_expanded = pd.concat(expanded_parts, ignore_index=True)

    # ---------- PLOTTING ----------
    unique_celltypes = df_expanded[celltype_key].unique()
    print(f"\nGenerating {len(unique_celltypes)} boxplots (one for each cell type).")

    for cell_type in unique_celltypes:
        plot_df = df_expanded[df_expanded[celltype_key] == cell_type].copy()
        if plot_df.empty:
            print(f"Skipping plot for {cell_type}: No data available.")
            continue

        # fixed-size figure for every cell type
        fig, ax = plt.subplots(figsize=figsize)

        # fixed margins so the axes width is constant across plots
        # (adjust these numbers if you want more/less space for labels)
        fig.subplots_adjust(left=0.22, right=0.95, bottom=0.35, top=0.88)

        # ----- region order -----
        if region_order is None:
            this_region_order = sorted(plot_df[region_key].dropna().unique())
        else:
            this_region_order = region_order

        palette_to_use = box_palette if box_palette is not None else 'Set2'

        # ----- boxplot -----
        ax = sns.boxplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            palette=palette_to_use,
            width=0.5,
            showfliers=False,
            linewidth=0.5,
            ax=ax
        )

        ylims = ax.get_ylim()

        # ----- stripplot -----
        ax = sns.stripplot(
            data=plot_df,
            x=region_key,
            y='proportion',
            order=this_region_order,
            size=1,
            jitter=True,
            dodge=False,
            edgecolor='black',
            linewidth=0.5,
            ax=ax
        )

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        # ----- add n above each box -----
        stats = (
            plot_df
            .groupby(region_key)['proportion']
            .agg(count='size', ymax='max')
            .reindex(this_region_order)
        )

        tick_locs = ax.get_xticks()
        tick_labels = [t.get_text() for t in ax.get_xticklabels()]
        x_pos_map = dict(zip(tick_labels, tick_locs))

        for region, row in stats.iterrows():
            if pd.isna(row['count']):
                continue
            n = int(row['count'])
            if region not in x_pos_map:
                continue
            x = x_pos_map[region]
            y_raw = (row['ymax'] * 1.05) if pd.notnull(row['ymax']) else 0.0
            y = min(y_raw, 0.98)

            ax.text(
                x,
                y,
                f"n={n}",
                ha="center",
                va="bottom",
                fontsize=7,
                color="black"
            )

        # ----- Mann–Whitney U and Levene tests between each pair of regions -----
        lines = []
        region_pairs = list(itertools.combinations(this_region_order, 2))

        for r1, r2 in region_pairs:
            g1 = plot_df.loc[plot_df[region_key] == r1, 'proportion'].dropna()
            g2 = plot_df.loc[plot_df[region_key] == r2, 'proportion'].dropna()

            if len(g1) < 2 or len(g2) < 2:
                continue

            stat_mw, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
            stat_lev, p_lev = levene(g1, g2, center='median')

            pair_header_added = False

            if p_mw <= 0.05:
                lines.append(f"{r1} vs {r2}:")
                lines.append(f"  MWU p={p_mw:.2e}")
                pair_header_added = True

            if p_lev <= 0.05:
                if not pair_header_added:
                    lines.append(f"{r1} vs {r2}:")
                lines.append(f"  Lev p={p_lev:.2e}")

        if lines:
            ax.text(
                0.72, 0.98,
                "\n".join(lines),
                transform=ax.transAxes,
                va='top',
                ha='left',
                fontsize=6,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8)
            )

        # ----- aesthetics -----
        ax.set_title(f"{cell_type}", fontsize=10)
        ax.set_xlabel(region_key.capitalize(), fontsize=9)
        ax.set_ylabel(f"Proportion of {cell_type}", fontsize=9)
        ax.set_ylim(0,0.1)
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.yticks(fontsize=8)

        safe_cell_type = cell_type.replace(" ", "_").replace("-", "_")
        plot_filename = os.path.join(
            savedir,
            f"Fig2_suppl_{safe_cell_type}_proportions_boxplot-Breslow.pdf"
        )

        try:
            fig.savefig(plot_filename)  # no bbox_inches='tight'
            print(f"  Plot for {cell_type} saved to: {plot_filename}")
        except Exception as e:
            print(f"  Error saving plot for {cell_type}: {e}")

        plt.close(fig)

In [106]:
order=['normal','precursor','MIS','invasive']

custom_colors = {
    "normal":   "#4782b6",
    "precursor":"#278b44",
    "MIS":      "#ea9ec5",
    "invasive": "#f1634b"
}

plot_proportions_by_celltype(
        df=data, 
        region_key='ROI_major_category', 
        celltype_key='lineage',
        sample_key='MELid',
        box_palette=custom_colors,
        region_order=order,
        figsize=(1.6, 3)
        )
        #savedir=r'C:\Users\tav9\HMS Dropbox\Tuulia Vallius\2024-Vallius-Novikov-Shi-Melanoma_PCA\3-Revision Nature Cancer March 2026\data\figure_panels')



Generating 6 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\365304103.py:78: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for SOX10+SOX9+NGFR-MART1+ saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9+NGFR_MART1+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\365304103.py:78: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




  Plot for SOX10+SOX9-NGFR-MART1+ saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9_NGFR_MART1+_proportions_boxplot-Breslow.pdf


...

  Plot for SOX10+SOX9+NGFR-MART1- saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9+NGFR_MART1__proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\365304103.py:78: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\anaconda3\envs\scimap_new\lib\site-packages\scipy\stats\_morestats.py:3310: RuntimeWarning:

invalid value encountered in scalar divide



  Plot for SOX10+SOX9-NGFR+MART1- saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9_NGFR+MART1__proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\365304103.py:78: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\anaconda3\envs\scimap_new\lib\site-packages\scipy\stats\_morestats.py:3310: RuntimeWarning:

invalid value encountered in scalar divide



  Plot for SOX10-SOX9+NGFR-MART1- saved to: celltype_proportion_plots\Fig2_suppl_SOX10_SOX9+NGFR_MART1__proportions_boxplot-Breslow.pdf


### plotting for SOX10+SOX9-NGFR+MART1-

In [107]:
## Modified script to allow breaking down of the y-axis

def plot_proportions_by_celltype(
    df,
    region_key='ROI_major_category',
    celltype_key='lineage',
    sample_key='MEL_id',
    region_order=None,
    box_palette=None,
    savedir='celltype_proportion_plots',
    # --- broken y-axis control ---
    bottom_ylim=None,        # e.g. (0, 0.2)
    top_ylim=None,           # e.g. (0.6, 1.0)
    figsize_single=(3, 3),
    figsize_broken=(3, 3.8),
    broken_height_ratios=(3, 1)  # NEW: relative heights (top, bottom)
):
    """
    Generates boxplots of proportions for each cell type, with:
      - Mann–Whitney U and Levene tests between region pairs (only p<=0.05 shown)
      - Samples with no ROI/celltype combinations filled with 0 for proportion/count/region_total.
      - Optional broken y-axis when bottom_ylim and top_ylim are given.
      - Optionally control relative vertical size of top vs. bottom of broken axis.
    """
    os.makedirs(savedir, exist_ok=True)
    sns.set_style("whitegrid")

    # ---------- EXPAND DF ----------
    all_celltypes = df[celltype_key].unique()
    expanded_parts = []

    for sample_id, sample_df in df.groupby(sample_key):
        sample_rois = sample_df[region_key].dropna().unique()
        full_index = pd.MultiIndex.from_product(
            [sample_rois, all_celltypes],
            names=[region_key, celltype_key]
        )
        sample_expanded = (
            sample_df
            .set_index([region_key, celltype_key])
            .reindex(full_index)
            .reset_index()
        )
        sample_expanded[sample_key] = sample_id

        if 'proportion' in sample_expanded.columns:
            sample_expanded['proportion'] = sample_expanded['proportion'].fillna(0)
        if 'count' in sample_expanded.columns:
            sample_expanded['count'] = sample_expanded['count'].fillna(0).astype(int)
        if 'region_total' in sample_expanded.columns:
            sample_expanded['region_total'] = sample_expanded['region_total'].fillna(0).astype(int)

        expanded_parts.append(sample_expanded)

    df_expanded = pd.concat(expanded_parts, ignore_index=True)

    # ---------- PLOTTING ----------
    unique_celltypes = df_expanded[celltype_key].unique()
    print(f"\nGenerating {len(unique_celltypes)} boxplots (one for each cell type).")

    use_broken_axis = (bottom_ylim is not None) and (top_ylim is not None)

    for cell_type in unique_celltypes:
        plot_df = df_expanded[df_expanded[celltype_key] == cell_type].copy()
        if plot_df.empty:
            print(f"Skipping plot for {cell_type}: No data available.")
            continue

        # ----- region order -----
        if region_order is None:
            this_region_order = sorted(plot_df[region_key].dropna().unique())
        else:
            this_region_order = region_order

        palette_to_use = box_palette if box_palette is not None else 'Set2'

        # ---------- FIGURE / AXES SETUP ----------
        if use_broken_axis:
            fig, (ax_top, ax_bottom) = plt.subplots(
                2, 1,
                sharex=True,
                figsize=figsize_broken,
                gridspec_kw={
                    "height_ratios": broken_height_ratios,  # <--- use parameter
                    "hspace": 0.05
                }
            )
            axes_for_plotting = [ax_top, ax_bottom]
        else:
            fig, ax = plt.subplots(figsize=figsize_single)
            axes_for_plotting = [ax]

        # ---------- BOXPLOT + STRIPPLOT ----------
        for a in axes_for_plotting:
            sns.boxplot(
                data=plot_df,
                x=region_key,
                y='proportion',
                order=this_region_order,
                palette=palette_to_use,
                width=0.5,
                showfliers=False,
                linewidth=0.5,
                ax=a
            )
            sns.stripplot(
                data=plot_df,
                x=region_key,
                y='proportion',
                order=this_region_order,
                size=1,
                jitter=True,
                dodge=False,
                edgecolor='black',
                linewidth=0.5,
                ax=a
            )

        if not use_broken_axis:
            ylims = ax.get_ylim()

        # remove legend
        for a in axes_for_plotting:
            leg = a.get_legend()
            if leg is not None:
                leg.remove()

        # ---------- n ABOVE EACH BOX ----------
        stats = (
            plot_df
            .groupby(region_key)['proportion']
            .agg(count='size', ymax='max')
            .reindex(this_region_order)
        )

        ax_for_n = axes_for_plotting[0]

        tick_locs = ax_for_n.get_xticks()
        tick_labels = [t.get_text() for t in ax_for_n.get_xticklabels()]
        x_pos_map = dict(zip(tick_labels, tick_locs))

        if use_broken_axis:
            y_max_for_n = top_ylim[1]
        else:
            y_max_for_n = ylims[1]

        for region, row in stats.iterrows():
            if pd.isna(row['count']):
                continue
            n = int(row['count'])
            if region not in x_pos_map:
                continue
            x = x_pos_map[region]
            y_raw = (row['ymax'] * 1.05) if pd.notnull(row['ymax']) else 0.0
            y = min(y_raw, y_max_for_n * 0.98)

            ax_for_n.text(
                x,
                y,
                f"n={n}",
                ha="center",
                va="bottom",
                fontsize=7,
                color="black"
            )

        # ---------- STATS TEXT ----------
        lines = []
        region_pairs = list(itertools.combinations(this_region_order, 2))

        for r1, r2 in region_pairs:
            g1 = plot_df.loc[plot_df[region_key] == r1, 'proportion'].dropna()
            g2 = plot_df.loc[plot_df[region_key] == r2, 'proportion'].dropna()

            if len(g1) < 2 or len(g2) < 2:
                continue

            stat_mw, p_mw = mannwhitneyu(g1, g2, alternative='two-sided')
            stat_lev, p_lev = levene(g1, g2, center='median')

            pair_header_added = False

            if p_mw <= 0.05:
                lines.append(f"{r1} vs {r2}:")
                lines.append(f"  MWU p={p_mw:.2e}")
                pair_header_added = True

            if p_lev <= 0.05:
                if not pair_header_added:
                    lines.append(f"{r1} vs {r2}:")
                lines.append(f"  Lev p={p_lev:.2e}")

        if lines:
            ax_for_stats = axes_for_plotting[0]
            ax_for_stats.text(
                0.72, 0.98,
                "\n".join(lines),
                transform=ax_for_stats.transAxes,
                va='top',
                ha='left',
                fontsize=6,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8)
            )

        # ---------- Y-LIMITS / BROKEN AXIS AESTHETICS ----------
        if use_broken_axis:
            ax_top.set_ylim(top_ylim)
            ax_bottom.set_ylim(bottom_ylim)

            ax_top.spines['bottom'].set_visible(False)
            ax_bottom.spines['top'].set_visible(False)

            ax_top.tick_params(labeltop=False)
            ax_bottom.tick_params(labeltop=False)

            # diagonal break marks
            d = .015
            kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False)
            ax_top.plot((-d, +d), (-d, +d), **kwargs)
            ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)

            kwargs = dict(transform=ax_bottom.transAxes, color='k', clip_on=False)
            ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs)
            ax_bottom.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

            ax = ax_top  # use top axis for title/y-label
        else:
            ax.set(ylim=ylims)

        # ---------- LABELS / TITLES ----------
        ax.set_title(f"{cell_type}", fontsize=10)
        ax.set_ylabel(f"Proportion of {cell_type}", fontsize=9)

        if use_broken_axis:
            ax_bottom.set_xlabel(region_key.capitalize(), fontsize=9)
        else:
            ax.set_xlabel(region_key.capitalize(), fontsize=9)

        # ticks
        if use_broken_axis:
            plt.setp(ax_bottom.get_xticklabels(), rotation=45, ha='right', fontsize=8)
            plt.setp(ax_top.get_xticklabels(), rotation=45, ha='right', fontsize=8)
            ax_top.tick_params(axis='y', labelsize=8)
            ax_bottom.tick_params(axis='y', labelsize=8)
        else:
            plt.xticks(rotation=45, ha='right', fontsize=8)
            plt.yticks(fontsize=8)

        fig.tight_layout(rect=(0, 0, 0.7, 1))

        safe_cell_type = cell_type.replace(" ", "_").replace("-", "_")
        plot_filename = os.path.join(
            savedir,
            f"Fig2_suppl_{safe_cell_type}_proportions_boxplot-Breslow.pdf"
        )

        try:
            fig.savefig(plot_filename)
            print(f"  Plot for {cell_type} saved to: {plot_filename}")
        except Exception as e:
            print(f"  Error saving plot for {cell_type}: {e}")

        plt.close(fig)

In [111]:

plot_proportions_by_celltype(
    df=data,
    region_key='ROI_major_category',
    celltype_key='lineage',
    sample_key='MELid',
    box_palette=custom_colors,
    region_order=order ,
    bottom_ylim=(0, 0.15),
    top_ylim=(0.6, 1.0),
    figsize_broken=(1.6, 2),
    broken_height_ratios=(1, 3)
)


Generating 6 boxplots (one for each cell type).


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:249: UserWarning:

This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.



  Plot for SOX10+SOX9+NGFR-MART1+ saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9+NGFR_MART1+_proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:249: UserWarning:

This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.



  Plot for SOX10+SOX9-NGFR-MART1+ saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9_NGFR_MART1+_proportions_boxplot-Breslow.pdf


...

  Plot for SOX10+SOX9+NGFR-MART1- saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9+NGFR_MART1__proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\anaconda3\envs\scimap_new\lib\site-packages\scipy\stats\_morestats.py:3310: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:249: UserWarning:

This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.



  Plot for SOX10+SOX9-NGFR+MART1- saved to: celltype_proportion_plots\Fig2_suppl_SOX10+SOX9_NGFR+MART1__proportions_boxplot-Breslow.pdf


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:95: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.


C:\Users\tav9\AppData\Local\anaconda3\envs\scimap_new\lib\site-packages\scipy\stats\_morestats.py:3310: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\tav9\AppData\Local\Temp\ipykernel_21324\70088029.py:249: UserWarning:

This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.



  Plot for SOX10-SOX9+NGFR-MART1- saved to: celltype_proportion_plots\Fig2_suppl_SOX10_SOX9+NGFR_MART1__proportions_boxplot-Breslow.pdf
